## Demultiplex GEX and sgRNA libraries

### Configuration

Edit the paths below to match your environment. All later cells reference these
variables; nothing else in this notebook contains a hard-coded path. The values
are also exported to the shell environment so the `%%bash` cells can use them.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR   = Path("/path/to/230328_scAtN")      # analysis root
RUN_DIR       = Path("/path/to/230324_A00201R_0856_AHK23YDMXY")  # Illumina run folder
CELLRANGER    = "/path/to/cellranger-6.0.1/cellranger"
TRANSCRIPTOME = "/path/to/refdata-gex-GRCh38-2020-A"
CONDA_ENV     = "/path/to/env"                     # conda env providing bcl2fastq
SLURM_PARTITION = "all"
N_SAMPLES     = 8                                  # GEX/CRISPR library pairs
INVESTIGATOR  = ""                                 # written into the sample sheets

DATA_DIR   = PROJECT_DIR / "data"
FASTQ_DIR  = DATA_DIR / "fastq"
STATS_DIR  = DATA_DIR / "stats"
COUNTS_DIR = PROJECT_DIR / "cellrangercounts"
LOG_DIR    = COUNTS_DIR / "logs"
FEATURE_REF = DATA_DIR / "CRISPR-feature-ref.csv"  # sgRNA feature reference

os.environ.update({
    "PROJECT_DIR": str(PROJECT_DIR),
    "RUN_DIR": str(RUN_DIR),
    "DATA_DIR": str(DATA_DIR),
    "FASTQ_DIR": str(FASTQ_DIR),
    "STATS_DIR": str(STATS_DIR),
    "COUNTS_DIR": str(COUNTS_DIR),
    "LOG_DIR": str(LOG_DIR),
    "CELLRANGER": CELLRANGER,
    "TRANSCRIPTOME": TRANSCRIPTOME,
    "FEATURE_REF": str(FEATURE_REF),
    "CONDA_ENV": CONDA_ENV,
    "SLURM_PARTITION": SLURM_PARTITION,
    "N_SAMPLES": str(N_SAMPLES),
})

def write(path, text):
    """Write text to path, creating parent directories as needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    print("Writing", path)


### GEX

In [ ]:
write(RUN_DIR / "gex.bc2fastq.csv", f"""
[Header],,,,,,,,,
IEMFileVersion,4,,,,,,,,
Investigator Name,{INVESTIGATOR},,,,,,,,
Experiment Name,scAtN,,,,,,,,
Date,3/28/23,,,,,,,,
Workflow,GenerateFASTQ,,,,,,,,
Application,FASTQ Only,,,,,,,,
Assay,Nextera,,,,,,,,
Description,"scAtN_GEX",,,,,,,,
Chemistry,Amplicon,,,,,,,,
,,,,,,,,,
[Reads],,,,,,,,,
50,,,,,,,,,
90,,,,,,,,,
,,,,,,,,,
[Settings],,,,,,,,,
ReverseComplement,0,,,,,,,,
,,,,,,,,,
[Data],,,,,,,,,
Sample_ID,Sample_Name,Sample_Plate,Sample_Well,I7_Index_ID,index,I5_Index_ID,index2,Sample_Project,Description
GEX_1,,,,SI-TT-A10,CGTGACATGC,SI-TT-A10,TTTAGACCAT,
GEX_2,,,,SI-TT-B10,GCCCGATGGA,SI-TT-B10,CTAGACGATT,
GEX_3,,,,SI-TT-C10,AGAATGGTTT,SI-TT-C10,TCCCACCCTC,
GEX_4,,,,SI-TT-D10,ATGCGAATGG,SI-TT-D10,CGACACTTGT,
GEX_5,,,,SI-TT-E10,CACAATCCCA,SI-TT-E10,TTGTGGATAT,
GEX_6,,,,SI-TT-F10,CCGGCAACTG,SI-TT-F10,TGTTAAACCG,
GEX_7,,,,SI-TT-G10,ACTTTACGTG,SI-TT-G10,AGGGCGTTCA,
GEX_8,,,,SI-TT-H10,TTATCTAGGG,SI-TT-H10,TAGAGCCTTT,
""")

In [ ]:
%%bash
module load Anaconda3
conda activate "$CONDA_ENV"
cd "$RUN_DIR"
sbatch \
    --partition "$SLURM_PARTITION" \
    --mem 100G \
    --cpus-per-task 10 \
    -o "$DATA_DIR/bcl2fastq.scAtN_gex.log" \
    <<EOF
#!/bin/bash

mkdir -p "$FASTQ_DIR/gex"
mkdir -p "$STATS_DIR/gex"
bcl2fastq \
    --use-bases-mask=Y26n*,I10,I10n*,Y90 \
    --create-fastq-for-index-reads \
    --minimum-trimmed-read-length=8 \
    --mask-short-adapter-reads=8 \
    --ignore-missing-positions \
    --ignore-missing-controls \
    --ignore-missing-filter \
    --ignore-missing-bcls \
    -r 6 -w 6 \
    -R "$RUN_DIR" \
    --output-dir="$FASTQ_DIR/gex" \
    --interop-dir="$STATS_DIR/gex" \
    --sample-sheet="$RUN_DIR/gex.bc2fastq.csv"

EOF

### gRNA

In [ ]:
write(RUN_DIR / "grna.bc2fastq.csv", f"""
[Header],,,,,,,,,
IEMFileVersion,4,,,,,,,,
Investigator Name,{INVESTIGATOR},,,,,,,,
Experiment Name,scAtN,,,,,,,,
Date,3/28/23,,,,,,,,
Workflow,GenerateFASTQ,,,,,,,,
Application,FASTQ Only,,,,,,,,
Assay,Nextera,,,,,,,,
Description,"scAtN_grna",,,,,,,,
Chemistry,Amplicon,,,,,,,,
,,,,,,,,,
[Reads],,,,,,,,,
50,,,,,,,,,
90,,,,,,,,,
,,,,,,,,,
[Settings],,,,,,,,,
ReverseComplement,0,,,,,,,,
,,,,,,,,,
[Data],,,,,,,,,
Sample_ID,Sample_Name,Sample_Plate,Sample_Well,I7_Index_ID,index,I5_Index_ID,index2,Sample_Project,Description
CRISPR_1,,,,SI-TT-A9,AAGTGGAGAG,SI-TT-A9,GTAACAGGAA,
CRISPR_2,,,,SI-TT-B9,TATTGAGGCA,SI-TT-B9,CACTTACCTG,
CRISPR_3,,,,SI-TT-C9,TATCAGCCTA,SI-TT-C9,AGGACGAAAC,
CRISPR_4,,,,SI-TT-D9,TGGTCCCAAG,SI-TT-D9,ACGCCAGAGG,
CRISPR_5,,,,SI-TT-E9,TGTCCCAACG,SI-TT-E9,TGGACATCGA,
CRISPR_6,,,,SI-TT-F9,GTCCCATCAA,SI-TT-F9,GTCACGTTCG,
CRISPR_7,,,,SI-TT-G9,CCGGAGGAAG,SI-TT-G9,AACATCCGCA,
CRISPR_8,,,,SI-TT-H9,AGAACTTAGA,SI-TT-H9,AAAGGACTCG,
""")

In [ ]:
%%bash
module load Anaconda3
conda activate "$CONDA_ENV"
cd "$RUN_DIR"
sbatch \
    --partition "$SLURM_PARTITION" \
    --mem 100G \
    --cpus-per-task 10 \
    -o "$DATA_DIR/bcl2fastq.scAtN_grna.log" \
    <<EOF
#!/bin/bash

mkdir -p "$FASTQ_DIR/grna"
mkdir -p "$STATS_DIR/grna"
bcl2fastq \
    --use-bases-mask=Y26n*,I10,I10n*,Y90 \
    --create-fastq-for-index-reads \
    --minimum-trimmed-read-length=8 \
    --mask-short-adapter-reads=8 \
    --ignore-missing-positions \
    --ignore-missing-controls \
    --ignore-missing-filter \
    --ignore-missing-bcls \
    -r 6 -w 6 \
    -R "$RUN_DIR" \
    --output-dir="$FASTQ_DIR/grna" \
    --interop-dir="$STATS_DIR/grna" \
    --sample-sheet="$RUN_DIR/grna.bc2fastq.csv"

EOF

## Run CellRanger Count

### Write .csv file to specify location of CRISPR and GEX libraries

In [ ]:
for i in range(1, N_SAMPLES + 1):
    write(DATA_DIR / f"GEX_{i}.csv", f"""fastqs,sample,library_type
{FASTQ_DIR}/gex,GEX_{i},Gene Expression
{FASTQ_DIR}/grna,CRISPR_{i},CRISPR Guide Capture
""")

### Run CellRanger

#### The following blocks are run per lane

The sgRNA feature reference (`CRISPR-feature-ref.csv`) is expected at
`FEATURE_REF` as defined in the configuration cell.

In [ ]:
%%bash
mkdir -p "$LOG_DIR"
cd "$LOG_DIR"
for i in $(seq 1 "$N_SAMPLES"); do
sbatch \
    -p "$SLURM_PARTITION" \
    -o "$LOG_DIR/sample_${i}_counts.log" \
    --mem=100G \
    --cpus-per-task=16 \
    <<EOF
#!/bin/bash

"$CELLRANGER" count --id=scAtN_sample_${i} \
    --transcriptome="$TRANSCRIPTOME" \
    --libraries="$DATA_DIR/GEX_${i}.csv" \
    --feature-ref="$FEATURE_REF" \
    --localcores=16 \
    --localmem=100
EOF
done

### Aggregate

Naming convention for matching samples (GEX and gRNA) must match the following: SampleName_S1_L002_R2_001.fastq.gz <br>
SampleName is used as the 'sample' value in the libraries.csv

Specify aggregate. Edit the below blocks to aggregate lanes for each donor and screen condition per design. The block below is shown for one donor/condition (pD2, samples 7 and 8) and is repeated for the remaining donors and conditions.

In [ ]:
write(PROJECT_DIR / "aggregate_pD2.csv", f"""sample_id,molecule_h5
scAtN_sample_7,{LOG_DIR}/scAtN_sample_7/outs/molecule_info.h5
scAtN_sample_8,{LOG_DIR}/scAtN_sample_8/outs/molecule_info.h5
""")

Aggregate

In [ ]:
%%bash
cd "$LOG_DIR"
sbatch \
    -p "$SLURM_PARTITION" \
    -o "$LOG_DIR/aggregate_pD2.log" \
    --mem=100G \
    --cpus-per-task=16 \
    <<EOF
#!/bin/bash

"$CELLRANGER" aggr --id=aggregate_pD2 --csv="$PROJECT_DIR/aggregate_pD2.csv" --normalize none --nosecondary
EOF